In [ ]:
!pip install colorama
import sqlite3
import sys
from colorama import Fore, Style, init
#NOTA: El trabajo es el tecer borrador, el primero anda (sin colorama) teniendo un producto guardado
#sin embargo el borrador dos y este que el final aparece un error
#(principalmente en la visualizacion, buscar producto, stock)
#en el colograma que no lo estoy pudiendo solucionar
#(mi padre que sabe de python y es tecnico cree que es algo de la actualizacion)


init(autoreset=True)

# para la base de datos (inventario.db)
DB_NAME = 'inventario.db'

# el siguiente codigo establece una conexion con DB_NAME como una constante y determinar una tupla simple
def conectar_db():
    """Establece la conexión con la base de datos."""
    try:
        conexion = sqlite3.connect(DB_NAME)
        conexion.row_factory = sqlite3.Row
        return conexion
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al conectar con la base de datos: {e}") #
        sys.exit(1)

# ahora tengo que crear un codigo para crear una tabla de los productos del almacen
def crear_tabla():
    """Crea la tabla 'productos' si no existe con el esquema requerido."""
    conn = conectar_db()
    cursor = conn.cursor()
    try:
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS productos (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                nombre TEXT NOT NULL,
                descripcion TEXT,
                cantidad INTEGER NOT NULL,
                precio REAL NOT NULL,
                categoria TEXT, -- MODIFICADO: Columna categoria reinsertada
                "clave del pais" TEXT,
                "clave de empresa" TEXT,
                "clave de producto" TEXT,
                "clave de verificador" TEXT
            )
        ''')
        conn.commit()
        print(f"{Fore.GREEN}Base de datos '{DB_NAME}' y tabla 'productos' listas.")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al crear la tabla: {e}")
    finally:
        conn.close()

# los codigos similares a la pre-entrega
def registrar_producto():
    """Permite ingresar y añadir un nuevo producto a la DB."""
    print(f"\n{Fore.CYAN}--- REGISTRAR NUEVO PRODUCTO ---{Style.RESET_ALL}")
    try:
        nombre = input("Nombre del producto (no nulo): ").strip()
        if not nombre:
            print(f"{Fore.RED}Error: El nombre del producto no puede estar vacío.")
            return
        descripcion = input("Descripción: ").strip()
        cantidad = int(input("Cantidad disponible (entero, no nulo): "))
        precio = float(input("Precio (real, no nulo): "))
        categoria = input("Categoría: ").strip()
        clave_pais = input("Clave del país (3 primeros dígitos del código de barras): ").strip()
        clave_empresa = input("Clave de empresa (4to al 7mo dígito): ").strip()
        clave_producto = input("Clave de producto (8vo al anteúltimo dígito): ").strip()
        clave_verificador = input("Clave de verificador (último dígito): ").strip()
    except ValueError:
        print(f"{Fore.RED}Error: Cantidad o Precio ingresados no son números válidos.")
        return
    conn = conectar_db()
    cursor = conn.cursor()
    try:
        cursor.execute('''
            INSERT INTO productos (nombre, descripcion, cantidad, precio, categoria, "clave del pais", "clave de empresa", "clave de producto", "clave de verificador")
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (nombre, descripcion, cantidad, precio, categoria, clave_pais, clave_empresa, clave_producto, clave_verificador))
        conn.commit()
        print(f"{Fore.GREEN} Producto '{nombre}' (ID: {cursor.lastrowid}) registrado con éxito.")
    except sqlite3.IntegrityError as e:
        print(f"{Fore.RED}Error de integridad al registrar el producto: {e}")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al registrar el producto: {e}")
    finally:
        conn.close()

# para la visualizacion utilizo una tabla
def visualizar_productos():
    """Muestra todos los productos registrados."""
    print(f"\n{Fore.CYAN}--- LISTA DE PRODUCTOS REGISTRADOS ---{Style.RESET_ALL}")
    try:
        with conectar_db() as conn:
            cursor = conn.cursor()
            cursor.execute("SELECT id, nombre, cantidad, precio, categoria FROM productos ORDER BY id")
            productos = cursor.fetchall()
            if not productos:
                print(f"{Fore.YELLOW}No hay productos registrados en el almacen.{Style.RESET_ALL}")
            print(f"{Fore.YELLOW}{'ID':<3} | {'Nombre':<22} | {'Categoría':<12} | {'Cantidad':<8} | {'Precio'}{Style.RESET_ALL}")
            print(f"{Fore.YELLOW}{'-'*3}|{'-'*24}|{'-'*14}|{'-'*10}|{'-'*8}{Style.RESET_ALL}")
            for prod in productos:
                producto_id, nombre, cantidad, precio, categoria = prod
                print(f"{producto_id:<3} | {nombre[:20]:<22} | {categoria[:10]:<12} | {cantidad:<8} | {precio:<8.2f}")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al visualizar productos: {e}{Style.RESET_ALL}")
    except NameError:
         print(f"{Fore.RED}Error: Asegúrate de importar e inicializar colorama (Fore, Style) y 'sqlite3'.{Style.RESET_ALL}")
    except Exception as e:
        print(f"{Fore.RED}Error inesperado: {e}{Style.RESET_ALL}")

def actualizar_producto():
    """Actualiza la cantidad y precio de un producto mediante su ID."""
    print(f"\n{Fore.CYAN}--- ACTUALIZAR PRODUCTO POR ID ---{Style.RESET_ALL}")
    try:
        valor_busqueda = int(input("Ingrese el ID del producto a actualizar: "))
        nueva_cantidad = int(input("Nueva cantidad disponible: "))
        nuevo_precio = float(input("Nuevo precio: "))
    except ValueError:
        print(f"{Fore.RED}Error: Entrada inválida (asegúrese de usar números para ID, Cantidad y Precio).")
        return
    conn = conectar_db()
    cursor = conn.cursor()
    try:
        query = "UPDATE productos SET cantidad = ?, precio = ? WHERE id = ?"
        cursor.execute(query, (nueva_cantidad, nuevo_precio, valor_busqueda))
        conn.commit()

        if cursor.rowcount > 0:
            print(f"{Fore.GREEN} Producto actualizado con éxito (ID: {valor_busqueda}).")
        else:
            print(f"{Fore.YELLOW} Producto con ID {valor_busqueda} no encontrado.")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al actualizar el producto: {e}")
    finally:
        conn.close()

# utilizando un similar metodo de actualizacion de metodo pero esta vez no actualiza, sino lo borra
def eliminar_producto():
    """Elimina un producto mediante su ID."""
    print(f"\n{Fore.CYAN}--- ELIMINAR PRODUCTO POR ID ---{Style.RESET_ALL}")
    try:
        producto_id = int(input("Ingrese el ID del producto a eliminar: "))
    except ValueError:
        print(f"{Fore.RED}Error: ID inválido.")
        return
    conn = conectar_db()
    cursor = conn.cursor()
    try:
        cursor.execute("DELETE FROM productos WHERE id = ?", (producto_id,))
        conn.commit()
        if cursor.rowcount > 0:
            print(f"{Fore.GREEN} Producto con ID {producto_id} eliminado con éxito.")
        else:
            print(f"{Fore.YELLOW} Producto con ID {producto_id} no encontrado.")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al eliminar el producto: {e}")
    finally:
        conn.close()

# para la busqueda del producto se necesita tener dos opciones por la forma de buscar
def buscar_producto():
    """Busca productos por ID, por nombre (parcial) o por categoría."""
    print(f"\n{Fore.CYAN}--- BUSCAR PRODUCTO ---{Style.RESET_ALL}")
    print("1. Buscar por ID")
    print("2. Buscar por Nombre (parcial)")
    print("3. Buscar por Categoría (opcional)")
    opcion = input(f"{Fore.BLUE}Seleccione una opción (1, 2 o 3):{Style.RESET_ALL} ").strip()
    conn = conectar_db()
    cursor = conn.cursor()
    productos = []
    query = ""
    params = ()
    try:
        if opcion == '1':
            prod_id = int(input("Ingrese el ID del producto: "))
            query = "SELECT id, nombre, categoria, cantidad, precio FROM productos WHERE id = ?"
            params = (prod_id,)
        elif opcion == '2':
            nombre = input("Ingrese parte del nombre: ").strip()
            query = "SELECT id, nombre, categoria, cantidad, precio FROM productos WHERE nombre LIKE ?"
            params = (f'%{nombre}%',)
        elif opcion == '3':
            categoria = input("Ingrese el nombre de la categoría: ").strip()
            query = "SELECT id, nombre, categoria, cantidad, precio FROM productos WHERE categoria LIKE ?"
            params = (f'%{categoria}%',)
        else:
            print(f"{Fore.RED}Opción inválida.")
            return
        cursor.execute(query, params)
        productos = cursor.fetchall()
        if productos:
            print(f"\n{Fore.GREEN}--- PRODUCTOS ENCONTRADOS ---{Style.RESET_ALL}")
            print(f"{Fore.YELLOW}{'ID':<3} | {'Nombre':<22} | {'Categoría':<12} | {'Cantidad':<8} | {'Precio'}{Style.RESET_ALL}")
            print(f"{Fore.YELLOW}{'-'*3}|{'-'*24}|{'-'*14}|{'-'*10}|{'-'*8}{Style.RESET_ALL}")
            for prod in productos:
                print(f"{prod['id']:<3} | {prod['nombre'][:20]:<22} | {prod['categoria'][:10]:<12} | {prod['cantidad']:<8} | {prod['precio']:<8.2f}")
        else:
            print(f"{Fore.YELLOW}No se encontraron productos.")
    except ValueError:
        print(f"{Fore.RED}Error: Entrada inválida (asegúrese de usar un número para el ID).")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al buscar productos: {e}")
    finally:
        conn.close()

# para conocer el stock de uno de los productos se debe verificar la cantidad del mismo
def reporte_bajo_stock():
    """Muestra productos con cantidad igual o inferior a un límite."""
    print(f"\n{Fore.CYAN}--- REPORTE DE BAJO STOCK ---{Style.RESET_ALL}")
    try:
        limite = int(input("Ingrese el límite máximo de cantidad (ej: 5): "))
    except ValueError:
        print(f"{Fore.RED}Error: Límite ingresado no es un número entero.")
        return
    conn = conectar_db()
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT id, nombre, cantidad, precio, categoria FROM productos WHERE cantidad <= ?", (limite,))
        productos = cursor.fetchall()
        if not productos:
            print(f"{Fore.GREEN}No hay productos con cantidad menor o igual a {limite}.")
            return
        print(f"\n{Fore.RED}--- PRODUCTOS CON STOCK BAJO (<= {limite}) ---{Style.RESET_ALL}")
        print(f"{Fore.YELLOW}{'ID':<3} | {'Nombre':<22} | {'Categoría':<12} | {'Cantidad':<8} | {'Precio'}{Style.RESET_ALL}")
        print(f"{Fore.YELLOW}{'-'*3}|{'-'*24}|{'-'*14}|{'-'*10}|{'-'*8}{Style.RESET_ALL}")
        for prod in productos:
            print(f"{Fore.RED}{prod['id']:<3} | {prod['nombre'][:20]:<22} | {prod['categoria'][:10]:<12} | {prod['cantidad']:<8} | {prod['precio']:<8.2f}{Style.RESET_ALL}")
    except sqlite3.Error as e:
        print(f"{Fore.RED}Error al generar el reporte: {e}")
    finally:
        conn.close()

# en esta parte es donde el usuario vera
def mostrar_menu():
    """Muestra el menú de opciones con colores."""
    print(f"\n{Fore.MAGENTA}=== Sistema Básico de Gestión del almacén ({DB_NAME}) ==={Style.RESET_ALL}")
    print(f"{Fore.YELLOW}1. Registrar (Agregar) producto{Style.RESET_ALL}")
    print(f"{Fore.YELLOW}2. Visualizar (Mostrar) productos{Style.RESET_ALL}")
    print(f"{Fore.YELLOW}3. Actualizar producto (por ID){Style.RESET_ALL}")
    print(f"{Fore.YELLOW}4. Eliminar producto (por ID){Style.RESET_ALL}")
    print(f"{Fore.YELLOW}5. Buscar producto (por ID, Nombre o Categoría){Style.RESET_ALL}")
    print(f"{Fore.YELLOW}6. Reporte de Bajo Stock{Style.RESET_ALL}")
    print(f"{Fore.RED}0. Salir{Style.RESET_ALL}")

def main():
    """Función principal que inicia la aplicación."""
    crear_tabla()
    while True:
        mostrar_menu()
        opcion = input(f"{Fore.BLUE}Seleccione una opción: {Style.RESET_ALL}").strip()
        if opcion == '1':
            registrar_producto()
        elif opcion == '2':
            visualizar_productos()
        elif opcion == '3':
            actualizar_producto()
        elif opcion == '4':
            eliminar_producto()
        elif opcion == '5':
            buscar_producto()
        elif opcion == '6':
            reporte_bajo_stock()
        elif opcion == '0':
            print(f"{Fore.RED}Saliendo del sistema. ¡Hasta luego!{Style.RESET_ALL}")
            break
        else:
            print(f"{Fore.RED}Opción no válida. Intente nuevamente.")
if __name__ == "__main__":
    main()

Base de datos 'inventario.db' y tabla 'productos' listas.

=== Sistema Básico de Gestión del almacén (inventario.db) ===
1. Registrar (Agregar) producto
2. Visualizar (Mostrar) productos
3. Actualizar producto (por ID)
4. Eliminar producto (por ID)
5. Buscar producto (por ID, Nombre o Categoría)
6. Reporte de Bajo Stock
0. Salir
